# BBC News Entertainment Subcategory Classifier

This notebook implements the full subcategory classification pipeline for the **Entertainment** category of the BBC News dataset, following the same architecture used for the Business category.

**Pipeline Overview:**
- Phase 1 — Data Ingestion
- Phase 2 — Text Cleaning
- Phase 3 — Vectorization (Jina AI)
- Phase 4 — UMAP + HDBSCAN Clustering
- Phase 5 — c-TF-IDF + KeyBERT Labeling
- Phase 5B — Verification Block
- Phase 6 — AI Subcategory Naming
- Phase 8 — Manual Noise Correction
- Phase 9 — Named Entity Recognition (NER) with April Events Extraction

## Project Overview
This notebook classifies 386 BBC Entertainment articles into meaningful subcategories, Beyond subcategory classification, the pipeline also performs Named Entity Recognition (NER) to extract media personalities and their roles, and identifies any events that took place or were scheduled to take place in April. using Jina Embeddings v5, UMAP dimensionality reduction, and HDBSCAN density-based clustering.

**Author:** BBC NLP Project 2
**Dataset:** 386 BBC Entertainment Articles
**Model:** Jina Embeddings v5 (text-small) with clustering adapter

---

## Phase 1: Data Ingestion & Structural Partitioning
In this phase we load all 386 articles from the `./entaintainment` folder into a structured DataFrame. Each article is split into its title and body, and metadata is tracked alongside the raw text.

In [1]:
# ============================================================
# PHASE 1: DATA INGESTION & STRUCTURAL PARTITIONING
# ============================================================

import os
import pandas as pd

# 1. Define the path to the entertainment folder
folder_path = './entertainment'

# 2. Initialize our data collection list
all_articles = []

# 3. Loop through each .txt file in the entertainment folder
for filename in sorted(os.listdir(folder_path)):
    if filename.endswith('.txt'):
        file_path = os.path.join(folder_path, filename)

        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read().strip()

        # 4. Split the file into title (first line) and body (rest)
        lines = content.split('\n')
        title = lines[0].strip()
        body = '\n'.join(lines[1:]).strip()

        # 5. Track metadata alongside raw text
        all_articles.append({
            'file_name': filename,
            'article_id': int(filename.replace('.txt', '')),
            'title': title,
            'raw_text': body,
            'article_length': len(body.split())
        })

# 6. Load into a structured DataFrame
df = pd.DataFrame(all_articles)

# 7. Quick sanity check
print(f" Total articles loaded: {len(df)}")
print(f" Columns: {df.columns.tolist()}")
print(f"\n Sample:")
df.head()

 Total articles loaded: 386
 Columns: ['file_name', 'article_id', 'title', 'raw_text', 'article_length']

 Sample:


,file_name,article_id,title,raw_text,article_length
0,001.txt,1,Gallery unveils interactive tree,A Christmas tree that can receive text message...,186
1,002.txt,2,Jarre joins fairytale celebration,French musician Jean-Michel Jarre is to perfor...,249
2,003.txt,3,Musical treatment for Capra film,The classic film It's A Wonderful Life is to b...,187
3,004.txt,4,Richard and Judy choose top books,The 10 authors shortlisted for a Richard and J...,208
4,005.txt,5,Poppins musical gets flying start,The stage adaptation of children's film Mary P...,174


## Phase 2: The Refinement Layer (Data Cleaning)

In this phase we clean all entertainment articles to prepare them for vectorization.
The cleaning process removes HTML tags and normalises whitespace to ensure
the text is consistent and ready for embedding.

In [16]:
# ============================================================
# PHASE 2: DATA CLEANING
# ============================================================
import re
import pandas as pd

def clean_article(text):

    # 1. HTML REMOVAL - confirmed present in raw BBC data
    text = re.sub(r'<[^>]+>', '', text)

    # 2. WHITESPACE CLEANUP
    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()

    return text

df = pd.read_csv('entertainment_subcategories_output.csv')

if 'raw_text' not in df.columns:
    df = df.rename(columns={'cleaned_text': 'raw_text'})

df['cleaned_text'] = df['raw_text'].apply(clean_article)
df = df.drop(columns=['raw_text'])

df.to_csv('entertainment_subcategories_output.csv', index=False)

print(f"Total articles: {len(df)}")
print(f"\nEntertainment cleaning complete!")

Total articles: 386

Entertainment cleaning complete!


## Phase 3: The Vectorization Layer (Jina-v5)

In this phase we convert all 386 entertainment articles into 1024-dimensional vector embeddings using the Jina AI API (`jina-embeddings-v5-text-small`, `text-matching` task). Embeddings are saved to `entertainment_embeddings.npy` to avoid repeated API calls.

In [3]:
# ============================================================
# PHASE 3: THE VECTORIZATION LAYER (JINA-V5) - RATE LIMIT FIX
# ============================================================

import requests
import numpy as np
import time
import os

JINA_API_KEY = "jina_0630e4e341144046ad7def8c563d0a7dcCRoxZqTtbVC15To_XeVA2-9PnKx"
JINA_URL = "https://api.jina.ai/v1/embeddings"

HEADERS = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {JINA_API_KEY}"
}

def get_embeddings(texts, batch_size=20):
    all_embeddings = []
    total_batches = (len(texts) + batch_size - 1) // batch_size

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        batch_num = (i // batch_size) + 1

        payload = {
            "model": "jina-embeddings-v5-text-small",
            "task": "text-matching",
            "normalized": True,
            "dimensions": 1024,
            "input": batch
        }

        success = False
        retries = 3

        while not success and retries > 0:
            try:
                response = requests.post(JINA_URL, headers=HEADERS, json=payload)
                response.raise_for_status()
                data = response.json()

                batch_embeddings = [item['embedding'] for item in data['data']]
                all_embeddings.extend(batch_embeddings)

                print(f" Batch {batch_num}/{total_batches} done — {len(all_embeddings)} articles embedded so far")
                success = True

                time.sleep(3)

            except Exception as e:
                retries -= 1
                print(f" Rate limit hit on batch {batch_num}, retrying in 10s... ({retries} retries left)")
                time.sleep(10)

        if not success:
            print(f" Failed on batch {batch_num} after all retries")
            break

    return np.array(all_embeddings)

# Only embed if embeddings don't already exist
if os.path.exists('entertainment_embeddings.npy'):
    print(" Embeddings already exist — skipping API call")
    embeddings = np.load('entertainment_embeddings.npy')
else:
    # Run on ALL 386 articles using title + content combined
    texts = (df['title'] + " " + df['cleaned_text']).tolist()
    print(f" Embedding {len(texts)} articles with Jina-v5 clustering adapter...\n")

    embeddings = get_embeddings(texts)

    # Save embeddings to disk permanently
    np.save('entertainment_embeddings.npy', embeddings)
    print(f"\n Embedding complete!")

print(f" Embedding matrix shape: {embeddings.shape}")
print(f"   → {embeddings.shape[0]} articles x {embeddings.shape[1]} dimensions")

 Embeddings already exist — skipping API call
 Embedding matrix shape: (386, 1024)
   → 386 articles x 1024 dimensions


## Phase 4: Dimensionality Reduction & Clustering

In this phase we reduce the 1024-dimensional embeddings to 3 dimensions using UMAP, then apply HDBSCAN to discover natural subcategory clusters. Results are saved to disk to avoid recomputation.

In [4]:
# ============================================================
# PHASE 4: DIMENSIONALITY REDUCTION & CLUSTERING
# ============================================================

import umap
import hdbscan
import numpy as np
import os

# Load embeddings from disk if available
if os.path.exists('entertainment_embeddings.npy'):
    print(" Loading saved embeddings...")
    embeddings = np.load('entertainment_embeddings.npy')
    print(f" Embeddings shape: {embeddings.shape}")

# Check if we already have saved cluster results
if os.path.exists('entertainment_cluster_labels.npy') and os.path.exists('entertainment_reduced_embeddings.npy'):
    
    print(" Loading saved cluster results...")
    cluster_labels = np.load('entertainment_cluster_labels.npy')
    reduced_embeddings = np.load('entertainment_reduced_embeddings.npy')
    confidence_scores = np.load('entertainment_confidence_scores.npy')
    
    df['subcategory_id'] = cluster_labels
    df['confidence_score'] = confidence_scores

    n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
    n_noise = list(cluster_labels).count(-1)

    print(f" Subcategories found: {n_clusters}")
    print(f"  Noise articles: {n_noise}")
    print(f" Clustered articles: {len(cluster_labels) - n_noise}")

else:
    
    # 1. UMAP
    print(" Running UMAP dimensionality reduction...")
    reducer = umap.UMAP(
        n_components=7,
        n_neighbors=20,
        min_dist=0.05,
        metric='cosine',
        random_state=42
    )
    reduced_embeddings = reducer.fit_transform(embeddings)
    print(f" UMAP complete! Shape: {reduced_embeddings.shape}\n")

    # 2. HDBSCAN
    print(" Running HDBSCAN clustering...")
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=9,
        min_samples=4,
        metric='euclidean',
        cluster_selection_method='leaf'
    )
    cluster_labels = clusterer.fit_predict(reduced_embeddings)
    confidence_scores = clusterer.probabilities_
    confidence_scores[cluster_labels == -1] = 0.0

    df['subcategory_id'] = cluster_labels
    df['confidence_score'] = confidence_scores

    # Save to disk permanently
    np.save('entertainment_cluster_labels.npy', cluster_labels)
    np.save('entertainment_reduced_embeddings.npy', reduced_embeddings)
    np.save('entertainment_confidence_scores.npy', confidence_scores)
    print(" Results saved to disk permanently!")

    n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
    n_noise = list(cluster_labels).count(-1)

    print(f" HDBSCAN complete!")
    print(f" Subcategories found: {n_clusters}")
    print(f"  Noise articles: {n_noise}")
    print(f" Clustered articles: {len(cluster_labels) - n_noise}")

# Cluster distribution
print(f"\n Articles per subcategory:")
print(df['subcategory_id'].value_counts().sort_index())

 Loading saved embeddings...
 Embeddings shape: (386, 1024)
 Loading saved cluster results...
 Subcategories found: 15
  Noise articles: 79
 Clustered articles: 307

 Articles per subcategory:
subcategory_id
-1     79
 0     22
 1     11
 2     53
 3     13
 4     19
 5     16
 6     10
 7      9
 8     28
 9     24
 10    29
 11    10
 12    41
 13    13
 14     9
Name: count, dtype: int64


## Phase 5: Thematic Labeling (c-TF-IDF)

In this phase we apply c-TF-IDF to extract the most distinctive keywords for each cluster. Words that appear frequently in one cluster but rarely in others are ranked highest, giving us meaningful thematic labels for each entertainment subcategory.

In [5]:
# ============================================================
# PHASE 5: THEMATIC LABELING (c-TF-IDF)
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
import numpy as np

# 1. Separate clustered articles (exclude noise label -1)
clustered_df = df[df['subcategory_id'] != -1].copy()

# 2. Group all articles in each cluster into one big document
cluster_documents = clustered_df.groupby('subcategory_id')['cleaned_text'].apply(
    lambda texts: ' '.join(texts)
).reset_index()

# 3. c-TF-IDF — Compare word frequencies ACROSS clusters
# This finds words that are unique to each cluster (not just common everywhere)
vectorizer = TfidfVectorizer(
    max_features=10000,
    stop_words='english',
    ngram_range=(1, 2),    # Include bigrams like "stock market", "interest rate"
    min_df=1
)

tfidf_matrix = vectorizer.fit_transform(cluster_documents['cleaned_text'])
feature_names = vectorizer.get_feature_names_out()

# 4. Extract top keywords per cluster
def get_top_keywords(tfidf_matrix, feature_names, n=10):
    top_keywords = []
    for i in range(tfidf_matrix.shape[0]):
        row = tfidf_matrix[i].toarray()[0]
        top_indices = row.argsort()[-n:][::-1]
        keywords = [feature_names[idx] for idx in top_indices]
        top_keywords.append(keywords)
    return top_keywords

top_keywords = get_top_keywords(tfidf_matrix, feature_names, n=10)

# 5. Display keywords per cluster
print(" Top keywords per subcategory:\n")
for i, row in cluster_documents.iterrows():
    cluster_id = row['subcategory_id']
    keywords = top_keywords[i]
    print(f"Cluster {cluster_id:2d} | {', '.join(keywords)}")

# 6. Add top 3 keywords as semantic label to DataFrame
cluster_documents['semantic_label'] = [
    ' | '.join(kws[:3]) for kws in top_keywords
]

# 7. Map labels back to main DataFrame
label_map = dict(zip(cluster_documents['subcategory_id'], cluster_documents['semantic_label']))
df['semantic_label'] = df['subcategory_id'].map(label_map)
df['semantic_label'] = df['semantic_label'].fillna('Noise')

print(f"\n Semantic labels assigned!")
print(f"\n Cluster ID → Semantic Label:")
for cluster_id, label in sorted(label_map.items()):
    print(f"  Cluster {cluster_id:2d} → {label}")

 Top keywords per subcategory:

Cluster  0 | ballet, theatre, musical, said, viotti, la, fenice, film, best, opera
Cluster  1 | music, chart, download, bpi, sales, industry, government, singles, piracy, phonographic
Cluster  2 | film, usd, gbp, box office, box, office, movie, films, said, oscar
Cluster  3 | bez, said, big brother, eviction, mccririck, celebrity, housemates, greer, evicted, stallone
Cluster  4 | said, forensic, fox, stern, viacom, mtv, network, tv, indecency, series
Cluster  5 | comedy, gervais, bbc, tv, little britain, best, said, series, channel, best tv
Cluster  6 | auction, sale, gallery, tapes, said, gbp, evans, nogar, tate, presley
Cluster  7 | book, novel, fiction, prize, award, judges, winner, year, won, whitbread
Cluster  8 | said, 50 cent, cent, mr, 50, rapper, court, police, rap, singer
Cluster  9 | film, festival, films, film festival, cinema, director, year, berlin, berlin film, dutch
Cluster 10 | film, aviator, best, swank, million dollar, dollar, dollar b

## Phase 5B: KeyBERT + Cluster Verification

In this phase we extract keywords for each article using KeyBERT, then print sample titles and cleaned text snippets for each cluster. This allows us to read the actual content before assigning subcategory names.

In [6]:
# ============================================================
# PHASE 5B: KEYBERT + CLUSTER VERIFICATION
# ============================================================
from keybert import KeyBERT
import warnings
warnings.filterwarnings('ignore')

# 1. KeyBERT keyword extraction
print(" Loading KeyBERT model...")
kw_model = KeyBERT()
print(" KeyBERT model loaded!\n")

print(" Extracting keywords for each article...")
def extract_keywords(text, n=5):
    try:
        keywords = kw_model.extract_keywords(
            str(text),
            keyphrase_ngram_range=(1, 2),
            stop_words='english',
            top_n=n
        )
        return ', '.join([kw[0] for kw in keywords])
    except:
        return ''

df['article_keywords'] = df['cleaned_text'].apply(extract_keywords)
print(f" Keywords extracted for all {len(df)} articles!\n")

# 2. Cluster verification — read before naming
print("=" * 70)
print("      CLUSTER VERIFICATION — READ BEFORE NAMING")
print("=" * 70)

for cluster_id in sorted(df[df['subcategory_id'] != -1]['subcategory_id'].unique()):
    cluster_articles = df[df['subcategory_id'] == cluster_id]
    
    print(f"\n{'='*70}")
    print(f"CLUSTER {cluster_id} — {len(cluster_articles)} articles")
    print(f"{'='*70}")
    
    # Sample articles — focus on cleaned_text
    samples = cluster_articles.sample(min(3, len(cluster_articles)), random_state=42)
    for i, (_, row) in enumerate(samples.iterrows(), 1):
        print(f"\n Title {i}: {row['title']}")
        print(f" Keywords: {row['article_keywords']}")
        snippet = ' '.join(str(row['cleaned_text']).split()[:80])
        print(f" Text: {snippet}...")
    
    print(f"\n  WHAT LABEL SHOULD THIS CLUSTER GET?")
    print(f"{'='*70}")

 Loading KeyBERT model...
 KeyBERT model loaded!

 Extracting keywords for each article...
 Keywords extracted for all 386 articles!

      CLUSTER VERIFICATION — READ BEFORE NAMING

CLUSTER 0 — 22 articles

 Title 1: Jarre joins fairytale celebration
 Keywords: andersen fairy, celebrate andersen, andersen denmark, christian andersen, danish royal
 Text: French musician Jean-Michel Jarre is to perform at a concert in Copenhagen to mark the bicentennial of the birth of writer Hans Christian Andersen. Denmark is holding a three-day celebration of the life of the fairy-tale author, with a concert at Parken stadium on 2 April. Other stars are expected to join the line-up in the coming months, and the Danish royal family will attend. "Christian Andersen's fairy tales are timeless and universal," said Jarre. "For all of us, at...

 Title 2: The Sound of Music is coming home
 Keywords: production musical, austria film, austrians song, musicals, musical
 Text: The original stage production of 

## Phase 6: Manual Label Assignment & Final Output

In this phase we assign professional subcategory names to each cluster based on the verification results from Phase 5B. The final output is saved to `entertainment_subcategories_output.csv`.

In [ ]:
# ============================================================
## Phase 6: Manual Label Assignment & Final Output
# ============================================================

# 1. AI-generated subcategory names for 16 clusters
ai_label_map = {
    0:  "Stage Performance",
    1:  "Music Downloads",
    2:  "Film Releases",
    3:  "Reality Television",
    4:  "Media Controversy",
    5:  "TV Comedy",
    6:  "Celebrity Sales",
    7:  "Literary Awards",
    8:  "Music Legal",
    9:  "Film Festivals",
    10: "Oscar Awards",
    11: "BAFTA Awards",
    12: "Music Artists",
    13: "Rock Legends",
    14: "Charity Music",
    -1: "Noise"
}

# 2. Apply confidence scores
df['confidence_score'] = confidence_scores
df.loc[df['subcategory_id'] == -1, 'confidence_score'] = 0.0

# 3. Apply AI labels
df['semantic_label'] = df['subcategory_id'].map(ai_label_map)

# 4. Build final output DataFrame cleanly
final_df = df[[
    'file_name',
    'article_id',
    'title',
    'cleaned_text',
    'subcategory_id',
    'semantic_label',
    'confidence_score',
    'article_keywords'
]].copy().reset_index(drop=True)

# 5. Save to CSV
final_df.to_csv('entertainment_subcategories_output.csv', index=False)

# 6. Summary
print("=" * 55)
print("    ENTERTAINMENT CLASSIFIER — FINAL RESULTS")
print("=" * 55)
print(f" Total Articles:        {len(final_df)}")
print(f" Subcategories Found:   {final_df[final_df['subcategory_id'] != -1]['subcategory_id'].nunique()}")
print(f" Clustered Articles:    {len(final_df[final_df['subcategory_id'] != -1])}")
print(f"  Noise Articles:        {len(final_df[final_df['subcategory_id'] == -1])}")
print(f" Avg Confidence:        {final_df[final_df['subcategory_id'] != -1]['confidence_score'].mean():.2f}")
print("=" * 55)
print(f"\n Label Distribution:")
print(final_df['semantic_label'].value_counts())
print(f"\n Saved to entertainment_subcategories_output.csv")

    ENTERTAINMENT CLASSIFIER — FINAL RESULTS
 Total Articles:        386
 Subcategories Found:   15
 Clustered Articles:    307
  Noise Articles:        79
 Avg Confidence:        0.89

 Label Distribution:
semantic_label
Noise                 79
Film Releases         53
Music Artists         41
Oscar Awards          29
Music Legal           28
Film Festivals        24
Stage Performance     22
Media Controversy     19
TV Comedy             16
Reality Television    13
Rock Legends          13
Music Downloads       11
Celebrity Sales       10
BAFTA Awards          10
Literary Awards        9
Charity Music          9
Name: count, dtype: int64

 Saved to entertainment_subcategories_output.csv


## Phase 8: Manual Noise Correction

In this phase we review all 79 noise articles and manually assign them to the most appropriate subcategory based on their content.

In [8]:
# Phase 8 - Noise Inspection: Show all noise articles with cleaned text
noise_df = df[df['semantic_label'] == 'Noise'].copy()

print(f"Total Noise Articles: {len(noise_df)}")
print("=" * 80)

for i, (idx, row) in enumerate(noise_df.iterrows(), 1):
    print(f"\n[{i}/{len(noise_df)}] article_id: {row['article_id']}")
    print(f"Title: {row['title']}")
    print(f"Cleaned Text (first 80 words):")
    words = str(row['cleaned_text']).split()
    snippet = ' '.join(words[:80])
    print(snippet)
    print("-" * 80)

Total Noise Articles: 79

[1/79] article_id: 6
Title: Bennett play takes theatre prizes
Cleaned Text (first 80 words):
The History Boys by Alan Bennett has been named best new play in the Critics' Circle Theatre Awards. Set in a grammar school, the play also earned a best actor prize for star Richard Griffiths as teacher Hector. The Producers was named best musical, Victoria Hamilton was best actress for Suddenly Last Summer and Festen's Rufus Norris was named best director. The History Boys also won the best new comedy title at the Theatregoers' Choice Awards. Partly based upon Alan
--------------------------------------------------------------------------------

[2/79] article_id: 8
Title: West End to honour finest shows
Cleaned Text (first 80 words):
The West End is honouring its finest stars and shows at the Evening Standard Theatre Awards in London on Monday. The Producers, starring Nathan Lane and Lee Evans, is up for best musical at the ceremony at the National Theatre. It is co

## Phase 8: Manual Noise Correction (Entertainment Category)

In this phase we manually assign the remaining noise articles that HDBSCAN 
could not confidently classify into a cluster. Each article was reviewed 
individually by reading the full article text and keywords before assigning 
the most appropriate subcategory label. This ensures zero noise articles 
remain in the final output and every article contributes meaningfully to 
the subcategory distribution.

The NER output is also updated after noise correction to ensure all 
downstream outputs reflect the correct subcategory labels.

In [9]:
# Phase 8 - Manual Noise Correction: Apply subcategory reassignments

noise_corrections = {
    6:   'Stage Performance',
    8:   'Stage Performance',
    30:  'Film Festivals',
    32:  'Film Festivals',
    40:  'Film Festivals',
    43:  'Film Releases',
    44:  'Film Releases',
    45:  'Film Releases',
    48:  'Film Releases',
    54:  'Film Releases',
    61:  'Film Releases',
    62:  'Film Festivals',
    63:  'Film Festivals',
    71:  'Film Releases',
    85:  'Film Releases',
    91:  'Film Festivals',
    104: 'Charity Music',
    107: 'Media Controversy',
    117: 'Music Artists',
    118: 'Music Artists',
    122: 'Music Artists',
    124: 'Rock Legends',
    138: 'Rock Legends',
    155: 'Music Legal',
    158: 'Rock Legends',
    161: 'Music Legal',
    162: 'Music Artists',
    172: 'Music Artists',
    182: 'Charity Music',
    183: 'Reality Television',
    188: 'Reality Television',
    194: 'Media Controversy',
    195: 'Charity Music',
    197: 'Media Controversy',
    202: 'Media Controversy',
    206: 'Reality Television',
    208: 'Music Artists',
    213: 'Reality Television',
    219: 'TV Comedy',
    220: 'BAFTA Awards',
    222: 'TV Comedy',
    227: 'Media Controversy',
    228: 'Film Releases',
    231: 'Media Controversy',
    244: 'Rock Legends',
    245: 'Music Artists',
    248: 'Music Legal',
    250: 'Music Artists',
    254: 'Music Artists',
    257: 'Rock Legends',
    259: 'Music Artists',
    260: 'Music Artists',
    261: 'Music Artists',
    263: 'Rock Legends',
    265: 'Stage Performance',
    268: 'Rock Legends',
    269: 'Music Artists',
    280: 'Oscar Awards',
    284: 'Media Controversy',
    300: 'Film Releases',
    301: 'Film Festivals',
    316: 'Film Releases',
    317: 'Stage Performance',
    319: 'Music Artists',
    324: 'Oscar Awards',
    327: 'Media Controversy',
    330: 'Film Releases',
    333: 'Film Releases',
    346: 'Stage Performance',
    347: 'Film Releases',
    349: 'Film Releases',
    356: 'Film Releases',
    365: 'TV Comedy',
    369: 'Oscar Awards',
    370: 'Film Festivals',
    371: 'Oscar Awards',
    375: 'Media Controversy',
    377: 'Oscar Awards',
    381: 'Film Releases',
}

# Apply corrections
for article_id, new_label in noise_corrections.items():
    df.loc[df['article_id'] == article_id, 'semantic_label'] = new_label

# Verify no noise remains
remaining_noise = df[df['semantic_label'] == 'Noise']
print(f"Remaining Noise Articles: {len(remaining_noise)}")
print()

# Show updated distribution
print("Updated Label Distribution:")
print(df['semantic_label'].value_counts())
print()
print(f"Total Articles: {len(df)}")

Remaining Noise Articles: 0

Updated Label Distribution:
semantic_label
Film Releases         70
Music Artists         55
Oscar Awards          34
Film Festivals        32
Music Legal           31
Media Controversy     28
Stage Performance     27
Rock Legends          20
TV Comedy             19
Reality Television    17
Charity Music         12
BAFTA Awards          11
Music Downloads       11
Celebrity Sales       10
Literary Awards        9
Name: count, dtype: int64

Total Articles: 386


# Phase 8: Manual Noise Correction — Final CSV Output

- Saves fully corrected Entertainment subcategories to CSV after Phase 8 noise reassignments
- All 79 noise articles have been manually reviewed by cleaned text and reassigned
- Zero noise articles remaining

Input  : df (with all Phase 8 corrections applied)
Output : entertainment_subcategories_output.csv

In [10]:
# Phase 6 Final Output - Save Entertainment CSV
output_df = df[[
    'file_name',
    'article_id',
    'title',
    'cleaned_text',
    'subcategory_id',
    'semantic_label',
    'confidence_score',
    'article_keywords'
]].copy()

output_df.to_csv('entertainment_subcategories_output.csv', index=False)

print("Entertainment pipeline complete.")
print(f"Total Articles: {len(output_df)}")
print(f"Subcategories: {output_df['semantic_label'].nunique()}")
print(f"Noise Remaining: {len(output_df[output_df['semantic_label'] == 'Noise'])}")
print(f"Avg Confidence: {output_df['confidence_score'].mean():.2f}")

Entertainment pipeline complete.
Total Articles: 386
Subcategories: 15
Noise Remaining: 0
Avg Confidence: 0.71


## Phase 9: Named Entity Recognition with April Events Extraction
In this phase we address the project's **Desired** requirement from the project brief:

> *"Identify documents and extract the named entities for media personalities, clearly identifying their jobs (e.g. Politicians, TV/Film Personalities, Musicians)"*

> *"Extract summaries of anything that took place or is/was scheduled to take place in April."*

We use spaCy to extract named entities from each article's combined `title` and `cleaned_text` for the **entertainment category** (369 articles). First we verify spaCy is installed and the English language model is available.

## Phase 9A: Named Entity Recognition — Persons, Organisations, Locations (Entertainment Category)
In this phase we extract named entities from each article's combined `title` and `cleaned_text` using spaCy's `en_core_web_lg` model across all 369 entertainment articles. Three entity types are extracted:

- **Named Persons** — validated through a false positive filter that removes award names, movie titles, single-name artists, month names, acronyms, and single-word misclassifications, ensuring only real two-word-minimum person names are retained
- **Named Organisations** — studios, record labels, broadcasters, award bodies, and other organisations mentioned
- **Named Locations** — countries, cities, venues, and regions identified as GPE or LOC entities

Each valid person is also classified by role using a context window of 80 characters surrounding the entity, matching against keyword lists for roles including Politician, TV/Film Personality, Musician, Author, Business Executive, Expert/Analyst, Sports Personality, Journalist, and Other. The entertainment classifier includes expanded keywords such as `starred`, `cast`, `role`, `screen`, `record`, `chart`, `tour`, `concert`, `author`, `novelist`, and `playwright`. Results: **363** articles with named persons, **358** with organisations, **347** with locations.

In [13]:
# ============================================================
# PHASE 9A: NER — PERSONS, ORGS, LOCATIONS (ENTERTAINMENT)
# ============================================================
import spacy
import pandas as pd

nlp = spacy.load('en_core_web_lg')
df = pd.read_csv('entertainment_subcategories_output.csv')
print(f" Loaded {len(df)} entertainment articles")

# ============================================================
# FALSE POSITIVE FILTER
# ============================================================
FALSE_POSITIVES = {
    # Movie/TV/Music titles
    'alexander', 'catwoman', 'rings', 'halo', 'titanic',
    'batman', 'superman', 'spiderman', 'matrix', 'avatar',
    'shrek', 'gladiator', 'braveheart', 'terminator',
    'eminem', 'beyonce', 'madonna', 'prince', 'cher',
    # Days/Months
    'monday', 'tuesday', 'wednesday', 'thursday', 'friday',
    'saturday', 'sunday', 'january', 'february', 'march',
    'april', 'may', 'june', 'july', 'august', 'september',
    'october', 'november', 'december',
    # Common misclassified words
    'lord', 'sir', 'mr', 'mrs', 'ms', 'dr', 'prof',
    'aol', 'sec', 'gm', 'ibm', 'bbc', 'itv',
    'oscar', 'bafta', 'grammy', 'brits',
}

def is_valid_person(name):
    if len(name) < 2:
        return False
    if name.lower() in FALSE_POSITIVES:
        return False
    if name.isupper():
        return False
    if not any(c.isalpha() for c in name):
        return False
    tokens = name.strip().split()
    if len(tokens) < 2:
        return False
    # filter possessives
    if name.endswith("'s") or name.endswith("s'"):
        return False
    # filter honorific-only entries like "Mr Doherty", "Ms Detakats"
    honorifics = {'mr', 'mrs', 'ms', 'dr', 'prof', 'dame', 'sir', 'lord'}
    if tokens[0].lower().rstrip('.') in honorifics and len(tokens) == 2:
        return False
    return True

# ============================================================
# KNOWN PERSON ROLE LOOKUP
# ============================================================
KNOWN_ROLES = {
    'Philip Roth':        'Author',
    'Muriel Spark':       'Author',
    'Abraham B Yehoshua': 'Author',
    'Clive Owen':         'TV/Film Personality',
    'Ricky Gervais':      'TV/Film Personality',
    'Tupac Shakur':       'Musician',
    'Rupert Adams':       'Spokesperson',
    'Franck Muller':      'Business Executive',
    'Scott Grogin':       'TV Producer',
    'Pete Winterbottom':  'Journalist',
    'Rafael Marques':     'Journalist',
}

# ============================================================
# PERSON ROLE CLASSIFIER
# ============================================================
def classify_role(person, context):
    # Check known persons first
    if person in KNOWN_ROLES:
        return KNOWN_ROLES[person]

    context = context.lower()
    if any(w in context for w in [
        'president', 'prime minister', 'minister', 'senator', 'mp ',
        'chancellor', 'governor', 'secretary of state', 'politician', 'parliament'
    ]):
        return 'Politician'
    elif any(w in context for w in [
        'lawyer', 'attorney', 'legal counsel', 'defence counsel',
        'defendant', 'prosecution', 'lawsuit', 'sued', 'trial',
        'verdict', 'indicted', 'charged', 'acquitted', 'solicitor', 'barrister'
    ]):
        return 'Lawyer'
    elif any(w in context for w in [
        'actor', 'actress', 'director', 'film', 'movie', 'tv', 'television',
        'presenter', 'host', 'comedian', 'starred', 'cast', 'role', 'screen',
        'oscar', 'bafta', 'box office', 'blockbuster', 'co-star', 'series',
        'sitcom', 'soap', 'drama', 'broadcast', 'channel 4', 'bbc one',
        'reality show', 'chat show', 'sketch', 'stand-up', 'stand up'
    ]):
        return 'TV/Film Personality'
    elif any(w in context for w in [
        'singer', 'musician', 'rapper', 'band', 'music', 'album', 'song',
        'record', 'chart', 'tour', 'concert', 'grammy', 'brit award',
        'guitarist', 'drummer', 'vocalist', 'producer', 'label', 'single',
        'track', 'gig', 'festival', 'rock', 'pop', 'jazz', 'classical', 'opera',
        'hip hop', 'hip-hop', 'mc ', 'emcee', 'rhyme', 'lyric',
        'rap', 'r&b', 'soul', 'reggae', 'punk', 'indie', 'dj '
    ]):
        return 'Musician'
    elif any(w in context for w in [
        'author', 'writer', 'novelist', 'poet', 'playwright', 'book',
        'novel', 'literary', 'booker', 'whitbread', 'published', 'fiction',
        'non-fiction', 'biography', 'memoir', 'story', 'chapter',
        'shortlist', 'longlist', 'literary award', 'prize', 'literature',
        'narrative', 'prose', 'verse', 'anthology', 'manuscript'
    ]):
        return 'Author'
    elif any(w in context for w in [
        'painter', 'sculptor', 'artist', 'artwork', 'gallery', 'exhibition',
        'installation', 'portrait', 'canvas', 'museum', 'tate', 'art world',
        'fashion designer', 'designer', 'couture', 'auction', 'portfolio',
        'collection', 'retrospective', 'print', 'illustration', 'sketch',
        'renaissance', 'modernist', 'contemporary art'
    ]):
        return 'Visual Artist'
    elif any(w in context for w in [
        'dancer', 'ballet', 'choreographer', 'dance', 'choreography',
        'royal ballet', 'performance', 'stage', 'theatre', 'musical',
        'ballerina', 'obituary', 'obituaries', 'passed away', 'died', 'death',
        'in memoriam', 'tribute', 'legacy', 'remembered'
    ]):
        return 'Performer'
    elif any(w in context for w in [
        'conductor', 'composer', 'orchestra', 'symphony', 'philharmonic',
        'opera house', 'concerto', 'sonata', 'score', 'overture', 'maestro'
    ]):
        return 'Conductor/Composer'
    elif any(w in context for w in [
        'ceo', 'chief executive', 'chairman', 'founder', 'executive',
        'managing director', 'mogul', 'tycoon', 'entrepreneur', 'businessm',
        'investor', 'billionaire', 'company', 'corporation', 'venture'
    ]):
        return 'Business Executive'
    elif any(w in context for w in [
        'analyst', 'economist', 'researcher', 'professor', 'scientist', 'expert',
        'academic', 'lecturer', 'scholar', 'doctor', 'psychologist'
    ]):
        return 'Expert/Analyst'
    elif any(w in context for w in [
        'coach', 'manager', 'player', 'footballer', 'athlete', 'champion',
        'tennis', 'cricket', 'rugby', 'sport', 'olympic', 'tournament',
        'league', 'match', 'fixture', 'transfer', 'squad', 'team'
    ]):
        return 'Sports Personality'
    elif any(w in context for w in [
        'journalist', 'reporter', 'editor', 'correspondent', 'critic',
        'columnist', 'broadcaster', 'commentator', 'presenter', 'anchor'
    ]):
        return 'Journalist'
    else:
        return 'Other'

# ============================================================
# NER EXTRACTION FUNCTION
# ============================================================
def extract_persons_orgs_locations(title, cleaned_text):
    combined = f"{title}. {cleaned_text}"
    doc = nlp(combined[:100000])

    persons = []
    person_roles = []
    orgs = []
    locations = []

    seen_persons = set()
    seen_orgs = set()
    seen_locations = set()

    for ent in doc.ents:

        if ent.label_ == 'PERSON':
            if ent.text not in seen_persons and is_valid_person(ent.text):
                seen_persons.add(ent.text)
                start = max(0, ent.start_char - 300)
                end = min(len(combined), ent.end_char + 300)
                context = combined[start:end]
                role = classify_role(ent.text, context)
                persons.append(ent.text)
                person_roles.append(f"{ent.text}: {role}")

        elif ent.label_ == 'ORG':
            if ent.text not in seen_orgs:
                seen_orgs.add(ent.text)
                orgs.append(ent.text)

        elif ent.label_ in ['GPE', 'LOC']:
            if ent.text not in seen_locations:
                seen_locations.add(ent.text)
                locations.append(ent.text)

    return {
        'named_persons':       ' | '.join(persons),
        'named_organisations': ' | '.join(orgs),
        'named_locations':     ' | '.join(locations),
        'person_roles':        ' | '.join(person_roles),
    }

# ============================================================
# RUN NER ON ALL ARTICLES
# ============================================================
print(" Running NER (Persons, Orgs, Locations)...")

named_persons       = []
named_organisations = []
named_locations     = []
person_roles        = []

for i, row in df.iterrows():
    result = extract_persons_orgs_locations(
        str(row['title']),
        str(row['cleaned_text'])
    )
    named_persons.append(result['named_persons'])
    named_organisations.append(result['named_organisations'])
    named_locations.append(result['named_locations'])
    person_roles.append(result['person_roles'])

    if (i + 1) % 50 == 0:
        print(f"   Processed {i + 1}/{len(df)} articles...")

df['named_persons']       = named_persons
df['named_organisations'] = named_organisations
df['named_locations']     = named_locations
df['person_roles']        = person_roles

print(f"\n Person/Org/Location NER complete!")
print(f" Articles with Persons:   {sum(1 for x in named_persons if x)}")
print(f" Articles with Orgs:      {sum(1 for x in named_organisations if x)}")
print(f" Articles with Locations: {sum(1 for x in named_locations if x)}")

# ============================================================

# ============================================================
# QUICK AUDIT — how many "Other" remain?
# ============================================================
other_count = df['person_roles'].str.contains('Other', na=False).sum()
print(f" Articles still containing 'Other': {other_count}")

 Loaded 386 entertainment articles
 Running NER (Persons, Orgs, Locations)...
   Processed 50/386 articles...
   Processed 100/386 articles...
   Processed 150/386 articles...
   Processed 200/386 articles...
   Processed 250/386 articles...
   Processed 300/386 articles...
   Processed 350/386 articles...

 Person/Org/Location NER complete!
 Articles with Persons:   378
 Articles with Orgs:      374
 Articles with Locations: 363
 Articles still containing 'Other': 0


## Phase 9B: April Events Extraction (Entertainment Category)
In this phase we extract April-related events from each article using spaCy's DATE entity recognition. For every article where a DATE entity containing "April" is detected in the combined `title` and `cleaned_text`, a 300-character contextual window surrounding the mention is captured as the event summary. This satisfies the project's Desired requirement of identifying events that took place or were scheduled to take place in April. Results: **21 out of 369** entertainment articles contained April events.

In [14]:
# ============================================================
# PHASE 9B: NER — APRIL EVENTS (ENTERTAINMENT CATEGORY)
# ============================================================

def extract_april_events(title, cleaned_text):
    combined = f"{title}. {cleaned_text}"
    doc = nlp(combined[:100000])

    april_events = []

    for ent in doc.ents:
        if ent.label_ == 'DATE':
            if 'april' in ent.text.lower():
                start = max(0, ent.start_char - 100)
                end = min(len(combined), ent.end_char + 200)
                april_events.append(combined[start:end].strip())

    return ' | '.join(april_events) if april_events else ''

print(" Extracting April events...")

april_events = []
for i, row in df.iterrows():
    result = extract_april_events(str(row['title']), str(row['cleaned_text']))
    april_events.append(result)

    if (i + 1) % 50 == 0:
        print(f"   Processed {i + 1}/{len(df)} articles...")

df['april_events'] = april_events

print(f"\n April Events extraction complete!")
print(f"  Articles with April Events: {sum(1 for x in april_events if x)}")

 Extracting April events...
   Processed 50/386 articles...
   Processed 100/386 articles...
   Processed 150/386 articles...
   Processed 200/386 articles...
   Processed 250/386 articles...
   Processed 300/386 articles...
   Processed 350/386 articles...

 April Events extraction complete!
  Articles with April Events: 22


## Phase 9C: Save Complete NER Output (Entertainment Category)
In this phase we consolidate all extracted NER columns — `named_persons`, `named_organisations`, `named_locations`, `person_roles`, and `april_events` — into the final dataframe and save to `entertainment_ner_output.csv`. A sample verification of the first article is printed to confirm all columns are correctly populated before proceeding to the role classifier improvement phase.

In [15]:
# ============================================================
# PHASE 9C: SAVE COMPLETE NER OUTPUT (ENTERTAINMENT CATEGORY)
# ============================================================

df['named_persons']       = named_persons
df['named_organisations'] = named_organisations
df['named_locations']     = named_locations
df['person_roles']        = person_roles
df['april_events']        = april_events

df.to_csv('entertainment_ner_output.csv', index=False)
print(" Saved to entertainment_ner_output.csv")
print(f" Columns: {df.columns.tolist()}")
print(f"\n Sample row 1:")
print(f" Persons:  {df['named_persons'].iloc[0]}")
print(f" Orgs:     {df['named_organisations'].iloc[0]}")
print(f" Location: {df['named_locations'].iloc[0]}")
print(f" Roles:    {df['person_roles'].iloc[0]}")
print(f"  April:    {df['april_events'].iloc[0]}")

print("\n" + "=" * 55)
print("    ENTERTAINMENT NER — FINAL SUMMARY")
print("=" * 55)
print(f" Total Articles:             {len(df)}")
print(f" Articles with Persons:      {sum(1 for x in named_persons if x)}")
print(f" Articles with Orgs:         {sum(1 for x in named_organisations if x)}")
print(f" Articles with Locations:    {sum(1 for x in named_locations if x)}")
print(f"  Articles with April Events: {sum(1 for x in april_events if x)}")
print("=" * 55)

 Saved to entertainment_ner_output.csv
 Columns: ['file_name', 'article_id', 'title', 'cleaned_text', 'subcategory_id', 'semantic_label', 'confidence_score', 'article_keywords', 'named_persons', 'named_organisations', 'named_locations', 'person_roles', 'april_events']

 Sample row 1:
 Persons:  Richard Wentworth | Tracey Emin | Henry Moore
 Orgs:     ArtWorks | Wentworth
 Location: London | Tate Britain | Norway
 Roles:    Richard Wentworth: Musician | Tracey Emin: Musician | Henry Moore: Musician
  April:    

    ENTERTAINMENT NER — FINAL SUMMARY
 Total Articles:             386
 Articles with Persons:      378
 Articles with Orgs:         374
 Articles with Locations:    363
  Articles with April Events: 22
